# Frozen Lake: Monte Carlo Methods

## Introduction

This notebook shows how to use simple MC methods to Gymnasium's Frozen lake environment, a small toy example for stochastic actions. 
Frozen lake involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. The player may not always move in the intended direction due to the slippery nature of the frozen lake.

The frozen lake environment is defined in `gymnasium` (not `gym-classics`) and only provides sample access. Here is the [environment description.](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)

The size of the problem, the randomness and the reward structure can be adjusted.

## Setup

You need:
* Gymnasium (see [Installation Instructions](../common/Setup_Gymnasium.ipynb))
* Patched `gym-classics-1.0.0+internal.rev1` or later (see [Installation instructions](../common/Setup_patched_gym_classics.ipynb))

In [1]:
import numpy as np
np.set_printoptions(precision=2)

In [2]:
import gymnasium as gym
import gym_classics
gym_classics.register('gymnasium')

In [3]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

## A Simple Example

### Create the Environment

**Note:** `success_rate` makes the ground slippery leading to stochastic behavior! From the [environment description](https://gymnasium.farama.org/environments/toy_text/frozen_lake/):

> `is_slippery=True`: If true the player will move in intended direction with probability specified by the 
> success_rate else will move in either perpendicular direction with equal probability in both directions.

I use here `success_rate=90.0/100.0` meaning that the ice is not very slippery and the agent will move with 90% probability into the direction of the chosen action. 

In [4]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make('FrozenLake-v1', 
               map_name="4x4", # can also be "8x8"
               is_slippery=True,
               success_rate=90.0/100.0,
               reward_schedule=(1, 0, 0),
               render_mode="rgb_array")

env_record = VideoWrapper(env, 'FL', render_fps=2)

Videos already exist, I remove them first!


/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/MC/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


We need simple code to run an episode.

In [5]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    Return = 0
    # run one episode
    for i in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        # step: execute an action in the environment
        observation_p, reward, terminated, truncated, info = env.step(action)

        if verbose:
            print (f"Step {i+1}: Obs {observation} -> Action {action} - > Reward {reward}, Obs' {observation_p}")

        observation = observation_p
        Return += reward

        # render the environment
        if render:
            env.render()

        if terminated:
            break
  
    if verbose:
        print(f"Episode Return: {Return}")
    
    return reward

### Random Agent

Check with am agent that moves randomly.

In [6]:
def random_agent_function(observation): 
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return env.action_space.sample()

In [7]:
run_episode(random_agent_function, env_record, max_steps = 100, verbose = True)
show(env_record)

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 4
Step 3: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 4: Obs 8 -> Action 1 - > Reward 0, Obs' 12
Episode Return: 0
Showing: ./videos/video_FL-episode-0.mp4


## Use MC Control with Exploring Starts 

We use here the implementation in `gym-classics`.

### Learn A Policy

In [8]:
from gym_classics.algorithms.monte_carlo_methods import MC_control_ES, MC_prediction

%time pol, Q = MC_control_ES(env, discount=1, n = 25000, max_episode_len= 30)
pol

MC Control: 100%|██████████| 25000/25000 [01:10<00:00, 353.19it/s]

CPU times: user 1min 9s, sys: 1.78 s, total: 1min 11s
Wall time: 1min 10s


array([1, 0, 1, 1, 1, 1, 0, 0, 2, 1, 1, 3, 0, 2, 2, 3])

Let's make the policy more readable and follow the layout of the problem.

In [9]:
def decode_action(action):
    return ['←','↓','→','↑'][int(action)]

def decode_policy(policy):
    return np.array([decode_action(a) for a in policy])

In [10]:
decode_policy(pol).reshape(4,4)

array([['↓', '←', '↓', '↓'],
       ['↓', '↓', '←', '←'],
       ['→', '↓', '↓', '↑'],
       ['←', '→', '→', '↑']], dtype='<U1')

Note that the agent sometimes runs into the wall to avoid the chance of falling into the lake.

MC Control returns the estimated Q-function. We can extract the value function to see the value for each state.

In [11]:
V = np.max(Q, axis = 1)
V.reshape(4,4)

array([[0.88, 0.84, 0.85, 0.88],
       [0.89, 0.89, 0.9 , 0.9 ],
       [0.93, 0.99, 0.93, 0.9 ],
       [0.88, 0.99, 0.99, 0.92]])

### Predict the Value Function

We can also use MC prediction (implemented in `gym-classics`). 

In [12]:
V = MC_prediction(env, pol, discount = 1, n =1000)
V.reshape(4,4)

MC Prediction: 100%|██████████| 1000/1000 [00:00<00:00, 12763.50it/s]


array([[0.87, 0.84,  nan,  nan],
       [0.87,  nan,  nan,  nan],
       [0.93, 0.99, 0.9 ,  nan],
       [ nan, 0.99, 0.99,  nan]])

Note that the samples always start from the start state and only follow the policy, so many states have no estimate. 

### Experiment with the Learned Policy

In [13]:
def policy_agent_function_generator(policy):
    def agent_function(obs):
        return policy[obs]
    return agent_function

In [14]:
policy_agent_function = policy_agent_function_generator(pol)

for i in range(10):
    G = run_episode(policy_agent_function, env_record, max_steps = 100)
    print(f"Episode {i}: Retrun={G}")
    show(env_record)


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 0: Retrun=1


Showing: ./videos/video_FL-episode-1.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 1: Retrun=1
Showing: ./videos/video_FL-episode-2.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 0
Step 2: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 3: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 4: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 5: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 6: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 7: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 2: Retrun=1
Showing: ./videos/video_FL-episode-3.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 3: Retrun=1
Showing: ./videos/video_FL-episode-4.mp4


...

Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 5: Retrun=1
Showing: ./videos/video_FL-episode-6.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 8
Step 5: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 6: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 7: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 8: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 6: Retrun=1
Showing: ./videos/video_FL-episode-7.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 7: Retrun=1
Showing: ./videos/video_FL-episode-8.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 8: Retrun=1
Showing: ./videos/video_FL-episode-9.mp4


Step 1: Obs 0 -> Action 1 - > Reward 0, Obs' 4
Step 2: Obs 4 -> Action 1 - > Reward 0, Obs' 8
Step 3: Obs 8 -> Action 2 - > Reward 0, Obs' 9
Step 4: Obs 9 -> Action 1 - > Reward 0, Obs' 13
Step 5: Obs 13 -> Action 2 - > Reward 0, Obs' 14
Step 6: Obs 14 -> Action 2 - > Reward 1, Obs' 15
Episode Return: 1
Episode 9: Retrun=1
Showing: ./videos/video_FL-episode-10.mp4


Perform evaluation with 100 random runs.

In [15]:
def policy_agent_function(obs):
    return pol[obs]

Gs = []

for i in range(100):
    G = run_episode(policy_agent_function, env, max_steps = 100, verbose = False)
    Gs.append(G)
   
print(Gs) 

print (f"Success rate: {np.mean(Gs)*100}%")

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Success rate: 91.0%


The success rate is rather high since we use a very small problem and a very high success rate.

## Reward Engineering

The Frozen Lake example lets us change the reward function. The default reward schedule:
* Reach goal: +1
* Reach hole: 0
* Reach frozen: 0 (walk on ice)


An important part of RL is to specify the reward so it is helpful for learning.  For the default setting, the agent initially no idea where to go and performs a random walk. 
All Q-values are initialized to 0 and do't change unless the agent randomly reaches the goa which may be very unlikely for large lakes. Lets change the reward structure
to make the agent scared of holes and a little more interested in walking on ice.

* Reach goal: +1 
* Reach hole: -.5
* Reach frozen: +0.01 (walk on ice)


In [16]:
env2 = gym.make('FrozenLake-v1', 
               map_name="4x4", # can also be "8x8"
               is_slippery=True,
               success_rate=90.0/100.0,
               reward_schedule=(1, -.5, 0.001),
               render_mode="rgb_array")

env_record2 = VideoWrapper(env2, 'FL2', render_fps=2)

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/MC/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [17]:
run_episode(random_agent_function, env2, max_steps=1000, verbose = True)

Step 1: Obs 0 -> Action 0 - > Reward 0.001, Obs' 0
Step 2: Obs 0 -> Action 2 - > Reward 0.001, Obs' 1
Step 3: Obs 1 -> Action 1 - > Reward -0.5, Obs' 5
Episode Return: -0.498


-0.5

Let's run a single episode and see how Q updates.

In [22]:
from gym_classics.algorithms.monte_carlo_methods import MC_control_ES, MC_prediction
pol2, Q2 = MC_control_ES(env2, discount=1, n = 1, max_episode_len= 30)
Q2

MC Control: 100%|██████████| 1/1 [00:00<00:00, 936.65it/s]


array([[ 0. , -0.5,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , -0.5],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. , -0.5,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. ,  0. ]])

Let's learn from more episodes

In [ ]:

%time pol2, Q2 = MC_control_ES(env2, discount=1, n = 10000, max_episode_len= 100)
pol2

MC Control: 100%|██████████| 10000/10000 [00:17<00:00, 566.18it/s]

CPU times: user 17.7 s, sys: 616 ms, total: 18.3 s
Wall time: 17.7 s


array([1, 3, 0, 3, 0, 3, 1, 1, 3, 1, 1, 3, 2, 2, 1, 2])

In [20]:
decode_policy(pol2).reshape(4,4)

array([['↓', '↑', '←', '↑'],
       ['←', '↑', '↓', '↓'],
       ['↑', '↓', '↓', '↑'],
       ['→', '→', '↓', '→']], dtype='<U1')

In [21]:
V = np.array([np.max(x) for x in Q2])
#V = MC_prediction(env, pol2, discount = 1, n =1000, verbose = False)
V.reshape(4,4)

array([[0.14, 0.13, 0.12, 0.13],
       [0.12, 0.11, 0.17, 0.15],
       [0.11, 0.57, 0.27, 0.12],
       [0.14, 0.58, 0.61, 0.11]])

This hopefully leads to better learning performance. But note, there is a chance for reward hacking! If we give the agent a large reward for walking around on the ice, then it will do that and avoid the goal to keep on getting reward.

Play with the reward function.


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)